Model

In [ ]:
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
import argparse
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
class Animal_Classifier(nn.Module):
  def __init__(self, num_classes):
    super(Animal_Classifier, self).__init__()
    self.conv_layers = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2)
    )

    self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((7,7)),
            nn.Flatten(),
            nn.Linear(128*7*7,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,num_classes)
        )

  def forward(self,x):
        x = self.conv_layers(x)
        x = self.classifier(x)
        return x

In [ ]:
if __name__ == "__main__":
  model = Animal_Classifier(num_classes=3)
  test_input = torch.randn(1, 3, 224, 224)
  output = model(test_input)
  print("Output shape:", output.shape)


Output shape: torch.Size([1, 3])


Dataset

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

class AnimalDataSet(Dataset):

    def __init__(
      self,
      data_dir,
      transforms = transforms.Compose(
        [
          transforms.Resize((128,128)),
          transforms.ToTensor(),
        ]
)):
        self.data = ImageFolder(data_dir, transform=transforms)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

    @property
    def classes(self):
        return self.data.classes




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
data_dir='/content/drive/MyDrive/folderting/datating/data/train'
data = AnimalDataSet(data_dir)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/folderting/datating/data/train'

In [ ]:
dataloader = DataLoader(data, batch_size=32, shuffle=True)

In [ ]:
for images, labels in dataloader:
  break

Training loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
transform = transforms.Compose([
transforms.Resize((128, 128))
,transforms.ToTensor(),
])

train_folder = '/content/drive/MyDrive/folderting/datating/data/train'
valid_folder = '/content/drive/MyDrive/folderting/datating/data/val'
test_folder = '/content/drive/MyDrive/folderting/datating/data/test'

train_dataset = AnimalDataSet(train_folder, transforms=transform)
val_dataset = AnimalDataSet(valid_folder, transforms=transform)
test_dataset = AnimalDataSet(test_folder, transforms=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader (test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Simple training loop
num_epochs = 5
train_losses, val_losses = [], []

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = Animal_Classifier(num_classes=3)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        # Move inputs and labels to the device
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * labels.size(0)
    train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(train_loss)

    # Validation phase
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            # Move inputs and labels to the device
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * labels.size(0)
    val_loss = running_loss / len(val_loader.dataset)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/{num_epochs} - Train loss: {train_loss}, Validation loss: {val_loss}")
    torch.save(model.state_dict(), '/content/drive/MyDrive/animal_model.pth')

In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label='Training loss')
plt.plot(val_losses, label='Validation loss')
plt.legend()
plt.title("Loss over epochs")
plt.show()

Classification

In [ ]:
import cv2
import torch
from torchvision import transforms
from PIL import Image

model = Animal_Classifier(num_classes=3)
model.load_state_dict(torch.load("/content/drive/MyDrive/animal_model.pth", map_location="cpu"))
model.eval()

classes = ["bear", "deer", "fox"]

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(img)

    input_tensor = transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(input_tensor)
        _, predicted = outputs.max(1)
        label = classes[predicted.item()]

    cv2.putText(frame, label, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Animal Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Load and preprocess the image
def preprocess_image(image_path, transform):
    image = Image.open(image_path).convert("RGB")
    return image, transform(image).unsqueeze(0)

# Predict using the model
def predict(model, image_tensor, device):
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.to(device)
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
    return probabilities.cpu().numpy().flatten()

# Visualization
def visualize_predictions(original_image, probabilities, class_names):
    fig, axarr = plt.subplots(1, 2, figsize=(14, 7))

    # Display image
    axarr[0].imshow(original_image)
    axarr[0].axis("off")

    # Display predictions
    axarr[1].barh(class_names, probabilities)
    axarr[1].set_xlabel("Probability")
    axarr[1].set_title("Class Predictions")
    axarr[1].set_xlim(0, 1)

    plt.tight_layout()
    plt.show()

# Example usage
# Define the device first
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = Animal_Classifier(num_classes=3)
model.load_state_dict(torch.load("/content/drive/MyDrive/animal_model.pth", map_location=device))
model.to(device) # Move the model to the correct device
model.eval()

classes = ["bear", "deer", "fox"]

test_image = "/content/drive/MyDrive/folderting/datating/data/test/american_black_bear/0BH737SIYQW8.jpg"
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

original_image, image_tensor = preprocess_image(test_image, transform)
probabilities = predict(model, image_tensor, device)

# Assuming dataset.classes gives the class names
# The `dataset` variable is defined in a previous cell, so it should be accessible.
class_names = train_dataset.classes
visualize_predictions(original_image, probabilities, class_names)


In [ ]:
from glob import glob
import os # Import os for os.path.join
import numpy as np # Ensure numpy is imported for np.random.choice

# Correcting the glob pattern to find image files within subdirectories
# Assuming images are directly in subfolders of test_folder
test_folder_path = '/content/drive/MyDrive/folderting/datating/data/test'
test_images = []
test_images.extend(glob(os.path.join(test_folder_path, '*', '*.jpg')))
test_images.extend(glob(os.path.join(test_folder_path, '*', '*.jpeg'))) # Add common extensions
test_images.extend(glob(os.path.join(test_folder_path, '*', '*.png')))

# Handle the case where there are fewer than 10 images
num_examples_to_pick = min(10, len(test_images))

if num_examples_to_pick == 0:
    print("No images found in the test directory to sample. Please check the path and image extensions.")
else:
    test_examples = np.random.choice(test_images, num_examples_to_pick, replace=False)

    for example in test_examples:
        original_image, image_tensor = preprocess_image(example, transform)
        probabilities = predict(model, image_tensor, device)

        # Using train_dataset.classes for class names
        class_names = train_dataset.classes
        visualize_predictions(original_image, probabilities, class_names)
